# AdaRSS: Personalized Career Mentorship Engine

**Lightweight Prototype** using TF-IDF + Logistic Regression. No downloads, runs on any CPU.

This notebook demonstrates the full vision:
1. **Classifies** skills as Enduring (0), Emergent (1), or Transient (2).
2. **Personalizes** advice based on a user's history (3 Personas tested below).

In [ ]:
import pandas as pd
import joblib
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
# 1. Load Data
df = pd.read_csv('data/sample_annotated.csv')
X = (df['job_title'] + ": " + df['skill']).values
y = df['label'].values

print(f"Loaded {len(X)} samples across multiple industries.")
print("Label distribution:")
print(df['label'].value_counts().sort_index())

In [ ]:
# 2. Train/Test Split & Vectorization
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=1500, stop_words='english', ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")
print(f"Feature matrix shape: {X_train_vec.shape}")

In [ ]:
# 3. Train Logistic Regression
model = LogisticRegression(multi_class='ovr', max_iter=1000, random_state=42)
model.fit(X_train_vec, y_train)
print("Model training complete.")

In [ ]:
# 4. Evaluate Performance
y_pred = model.predict(X_test_vec)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=['Enduring', 'Emergent', 'Transient'])

print(f"✅ Validation Accuracy: {acc:.2%}\n")
print("Confusion Matrix:")
print(cm)
print("\nClassification Report:")
print(report)

In [ ]:
# 5. Save Model & Vectorizer for inference
joblib.dump(model, './api/model/adarss-baseline.pkl')
joblib.dump(vectorizer, './api/model/adarss-vectorizer.pkl')
print("💾 Models saved locally.")

In [ ]:
# 6. Helper: Classify a single text
def classify_skill_text(text):
    vec = vectorizer.transform([text])
    pred = model.predict(vec)[0]
    mapping = {0: "Enduring", 1: "Emergent", 2: "Transient"}
    return pred, mapping[pred]

# Test it
print(classify_skill_text("Frontend Developer: React hooks"))  # Should be Transient

## 🧠 The Personalization Engine

Below is the core logic that decides between **Upscaling**, **Broadening**, or **Changing** profession based on the user's history and the target skill.

In [ ]:
class PersonalizationEngine:
    def __init__(self, history_text):
        self.history = history_text.lower()
        self.history_keywords = set(re.findall(r'\b[a-z]{3,}\b', self.history))
    
    def _calculate_overlap(self, target_skill):
        target_words = set(re.findall(r'\b[a-z]{3,}\b', target_skill.lower()))
        overlap = len(self.history_keywords.intersection(target_words))
        if target_skill.lower() in self.history:
            overlap += 5
        return overlap

    def generate_advice(self, target_skill):
        overlap_score = self._calculate_overlap(target_skill)
        pred, durability = classify_skill_text(f"Generic Worker: {target_skill}")
        
        # Domain matching heuristic
        tech = {'python','java','sql','docker','kubernetes','react','django','api','linux','devops','aws','gcp'}
        health = {'nurse','patient','clinical','epic','emr','medical','triage'}
        finance = {'accounting','gaap','audit','quickbooks','finance','excel'}
        
        hist_set = self.history_keywords
        is_hist_tech = bool(hist_set.intersection(tech))
        is_hist_health = bool(hist_set.intersection(health))
        is_hist_finance = bool(hist_set.intersection(finance))
        
        target_words = set(re.findall(r'\b[a-z]{3,}\b', target_skill.lower()))
        is_tar_tech = bool(target_words.intersection(tech))
        is_tar_health = bool(target_words.intersection(health))
        is_tar_finance = bool(target_words.intersection(finance))
        
        domain_match = ((is_hist_tech and is_tar_tech) or 
                        (is_hist_health and is_tar_health) or 
                        (is_hist_finance and is_tar_finance))
        
        if overlap_score >= 3 and domain_match:
            advice_type = "Continuous Upscaling"
            desc = "✅ Your background aligns perfectly. You already have the prerequisites."
            roadmap = f"Mastery path: Advanced concepts in {target_skill}, applied projects, certification. Since {target_skill} is {durability}, it's a solid long-term bet."
        elif overlap_score >= 1 and not domain_match:
            advice_type = "Career Broadening"
            desc = f"🌉 Your history doesn't directly match {target_skill}, but you have transferable foundational skills."
            roadmap = f"Leverage your current domain expertise (e.g., {self.history[:40]}...) to enter {target_skill} space. {target_skill} is classified as {durability}."
        else:
            advice_type = "Change of Profession"
            desc = f"⚠️ This is a clean pivot. {target_skill} has minimal overlap with your history."
            roadmap = f"Ground-zero approach: 1. Enroll in bootcamp, 2. Build portfolio, 3. Internship. Note: {target_skill} is currently {durability}. Plan for 12-24 months."
            
        return {
            "type": advice_type,
            "description": desc,
            "roadmap": roadmap,
            "durability": durability
        }

## 👥 Demo: 3 Distinct Personas

Let's test the engine with Aisha (Backend Dev), James (Nurse), and Maria (Accountant).

In [ ]:
print("="*70)
print("🧑‍💼 ADA RSS PERSONALIZED CAREER ADVICE (DEMO)")
print("="*70)

# Persona A: Continuous Upscaling
print("\n--- 👩‍💻 Persona A: Aisha (Backend Dev) ---")
engine_a = PersonalizationEngine("5 years Python, Django, REST APIs, SQL databases.")
advice_a = engine_a.generate_advice("Kubernetes")
print(f"History: 5 years Python, Django, REST APIs, SQL databases.")
print(f"Target Skill: Kubernetes")
print(f"🏷️ Classification: {advice_a['durability']}")
print(f"📌 Advice Type: {advice_a['type']}")
print(f"💬 {advice_a['description']}")
print(f"🗺️ {advice_a['roadmap']}")

# Persona B: Career Broadening
print("\n--- 👨‍⚕️ Persona B: James (Registered Nurse) ---")
engine_b = PersonalizationEngine("10 years ER Nurse, experienced with Epic Systems, patient triage.")
advice_b = engine_b.generate_advice("Health Informatics")
print(f"History: 10 years ER Nurse, Epic Systems, patient triage.")
print(f"Target Skill: Health Informatics")
print(f"🏷️ Classification: {advice_b['durability']}")
print(f"📌 Advice Type: {advice_b['type']}")
print(f"💬 {advice_b['description']}")
print(f"🗺️ {advice_b['roadmap']}")

# Persona C: Change of Profession
print("\n--- 👩‍💼 Persona C: Maria (Accountant) ---")
engine_c = PersonalizationEngine("8 years Financial Audit, QuickBooks, GAAP compliance.")
advice_c = engine_c.generate_advice("UI/UX Design")
print(f"History: 8 years Financial Audit, QuickBooks, GAAP compliance.")
print(f"Target Skill: UI/UX Design")
print(f"🏷️ Classification: {advice_c['durability']}")
print(f"📌 Advice Type: {advice_c['type']}")
print(f"💬 {advice_c['description']}")
print(f"🗺️ {advice_c['roadmap']}")

print("\n" + "="*70)
print("✨ Prototype ready. Next step: Scale to 50k+ job postings.")
print("="*70)